In [0]:
%sql
-- Check if any duplicate PK for cust_info 
SELECT cst_id, COUNT(*) 
FROM bronze.crm_cust_info 
GROUP BY cst_id 
HAVING COUNT(*)>1 OR cst_id IS NULL; 
-- Results: There're some duplicate and null cst_id  

-- Solution: Using window function to get the latest record for each duplicate cst_id 
SELECT * FROM 
(
SELECT *, ROW_NUMBER() OVER (PARTITION BY cst_id ORDER BY cst_create_date DESC) AS flag_last 
FROM bronze.crm_cust_info
WHERE cst_id IS NOT NULL
)
WHERE flag_last = 1 ; 

-- Check unwanted space, Data Standardization & Consistency 
SELECT 
cst_id, 
cst_key,
TRIM(cst_firstname) AS cst_firstname,
TRIM(cst_lastname) AS cst_lastname, 
cst_marital_status,
CASE WHEN UPPER(TRIM(cst_gndr)) = 'F' THEN 'Female'
     WHEN UPPER(TRIM(cst_gndr)) = 'M' Then 'Male'
     ELSE 'n/a'
END cst_gndr,
CASE  WHEN UPPER(TRIM(cst_marital_status)) = 'M' THEN 'Married'
      WHEN UPPER(TRIM(cst_marital_status)) = 'S' THEN 'Single'
      ELSE 'n/a'
END cst_marital_status, 
DATE(cst_create_date)
FROM 
(
  SELECT * FROM 
(
SELECT *, ROW_NUMBER() OVER (PARTITION BY cst_id ORDER BY cst_create_date DESC) AS flag_last 
FROM bronze.crm_cust_info
WHERE cst_id IS NOT NULL
)
WHERE flag_last = 1 
)

In [0]:
%sql
-- Check if any duplicate PK for cust_info silver  
SELECT cst_id, COUNT(*) 
FROM silver.crm_cust_info 
GROUP BY cst_id 
HAVING COUNT(*)>1 OR cst_id IS NULL; 

-- Check for unwated spcaces 
SELECT cst_firstname, cst_lastname, cst_marital_status, cst_gndr
FROM silver.crm_cust_info 
WHERE cst_firstname != TRIM(cst_firstname) 
OR    cst_lastname != TRIM(cst_lastname) 
OR    cst_marital_status != TRIM(cst_marital_status)
OR    cst_gndr != TRIM(cst_gndr); 

-- Data Standardization & Consistency 
Select DISTINCT cst_gndr, cst_marital_status
FROM silver.crm_cust_info 

In [0]:
%sql
-- Check if there is any null and duplicates PK for prd_info table
SELECT prd_id, COUNT(*)
FROM bronze.crm_prd_info
GROUP BY prd_id 
HAVING COUNT(*) > 1 OR prd_id IS NULL;
-- Result: No duplicates or Null PK 

-- check unwanted spaces
SELECT prd_nm FROM bronze.crm_prd_info
WHERE prd_nm != TRIM(prd_nm);
-- result: no unwated space 

-- check for null or negative number 
SELECT prd_cost 
FROM bronze.crm_prd_info 
WHERE prd_cost < 0 OR prd_cost IS NULL ;
-- Solution: replace with zero 
SELECT COALESCE(prd_cost, 0) AS prd_cost
FROM bronze.crm_prd_info;

-- check data standardization 
SELECT DISTINCT prd_line FROM bronze.crm_prd_info;
-- Solution: 
SELECT 
CASE WHEN TRIM(UPPER(prd_line))= 'M' THEN 'Mountain'
     WHEN TRIM(UPPER(prd_line))= 'R' THEN 'Road'
     WHEN TRIM(UPPER(prd_line))= 'T' THEN 'Touring'
     WHEN TRIM(UPPER(prd_line))= 'S' THEN 'other Sales'
     ELSE 'n/a'
END prd_line
FROM bronze.crm_prd_info; 

-- Separeate the prd_key column into cat_id and prd_key, so we can join prd_info and px_info tables with FK is cat_id 
SELECT prd_id, 
prd_key, 
REPLACE(SUBSTRING(prd_key, 1, 5), '-','_') AS cat_id,  -- Data Transformation to match with cat_id in px_cat_g1v2 table
SUBSTRING(prd_key, 7, length(prd_key)) AS prd_key,
prd_nm,
prd_cost,
prd_line,
prd_start_dt,
prd_end_dt 
FROM bronze.crm_prd_info 
WHERE REPLACE(SUBSTRING(prd_key, 1, 5), '-','_') NOT IN (
  SELECT DISTINCT id from bronze.erp_px_cat_g1v2
); -- filter some unmatched cat_id 
-- result: only CO_PE cat_id is not matched with px_cat_g1v2 table 

-- Check data quality for Date 
SELECT * FROM bronze.crm_prd_info
WHERE prd_start_dt  > prd_end_dt;

-- Problem: start_dt > end_dt, Cannot swap the dates because records with the same prd_key may have overlapping periods.
-- Solution: Use window function LEAD() 
SELECT prd_key, 
CAST(prd_start_dt AS DATE), 
LEAD(CAST(prd_start_dt AS DATE)) OVER (PARTITION BY prd_key ORDER BY prd_start_dt) -1 AS prd_end_dt 
FROM bronze.crm_prd_info 

In [0]:
%sql
-- check if the sls_prd_key match with prd_key in prd_info 
SELECT * FROM bronze.crm_sales_details
WHERE sls_prd_key NOT IN 
(  
  SELECT prd_key FROM silver.crm_prd_info
  ); 
-- Result: No unmatched data 

-- check if cust_id in sales table match cust_id in cust_info table 
SELECT * FROM bronze.crm_sales_details 
WHERE sls_cust_id NOT IN(
  SELECT cst_id FROM silver.crm_cust_info
);
-- Result: No unmatched data 

--check unwanted space
SELECT sls_ord_num, sls_prd_key 
FROM bronze.crm_sales_details 
WHERE sls_ord_num != TRIM(sls_ord_num)
OR sls_prd_key != TRIM(sls_prd_key);  
-- no unwanted space 

-- check data consistency, sales, quantity, price cannot be negative, sales = quantity * price 
SELECT
sls_sales AS old_sls_sales,
sls_price AS old_sls_price,
CASE WHEN sls_sales IS NULL OR sls_sales <= 0 OR sls_sales != sls_quantity * ABS(sls_price) 
        THEN sls_quantity * sls_price
     ELSE sls_sales 
END sls_sales, 

CASE WHEN sls_price = 0 OR sls_price IS NULL 
        THEN sls_sales / sls_quantity
     WHEN sls_price <0
        then abs(sls_price)
     ELSE sls_price
end sls_price,
sls_quantity      
from bronze.crm_sales_details;
 
 -- rules: 
-- if sales is negative, zero or null, derive it using quantity and price
-- price is zero or null, caculate it using sales , quantity 
-- price is negative, convert to positive value 

-- check invalid date/boundaries of date range 
SELECT 
NULLIF(sls_order_dt, 0),
sls_ship_dt
FROM bronze.crm_sales_details
WHERE sls_order_dt <=0 
OR len(sls_order_dt) !=8
OR sls_order_dt > 20500101
OR sls_order_dt < 19000101; --when your business start 

SELECT * FROM bronze.crm_sales_details
WHERE sls_order_dt > sls_ship_dt 
OR sls_order_dt > sls_due_dt; --order date cannot be later than ship date or due date -- no invalid date 

SELECT  
       CASE WHEN sls_order_dt = 0 OR len(sls_order_dt) != 8 THEN NULL        -- Data transformation for date format 
            ELSE TO_DATE(CAST(sls_order_dt AS VARCHAR(50)), 'yyyyMMdd')
       END sls_order_dt,
       CASE WHEN sls_due_dt = 0 OR len(sls_due_dt) != 8 THEN NULL  
            ELSE TO_DATE(CAST(sls_due_dt AS VARCHAR(50)), 'yyyyMMdd')
       END sls_due_dt,
       CASE WHEN sls_ship_dt = 0 OR len(sls_ship_dt) !=8 THEN NULL 
            ELSE TO_DATE(CAST(sls_ship_dt AS VARCHAR(50)), 'yyyyMMdd')
       END sls_ship_dt
FROM bronze.crm_sales_details

In [0]:
%sql
-- check for any unmatched data 
-- result: no unmatched data 
SELECT
CASE WHEN CID LIKE 'NAS%' THEN SUBSTRING(CID, 4, len(CID))
     ELSE CID
END CID,
CASE WHEN CAST (BDATE AS DATE) > CURRENT_DATE() THEN NULL  
     ELSE CAST (BDATE AS DATE)
END BDATE,
GEN 
FROM bronze.erp_cust_az12
WHERE CASE WHEN CID LIKE 'NAS%' THEN SUBSTRING(CID, 4, len(CID))
     ELSE CID
END  NOT IN (SELECT DISTINCT cst_key FROM silver.crm_cust_info); 

-- check bdate out of range 
SELECT * FROM bronze.erp_cust_az12
WHERE BDATE > CURRENT_DATE();

-- Data standardization & Consistency 
SELECT DISTINCT gen 
FROM bronze.erp_cust_az12;
-- Solution
SELECT
CASE WHEN CID LIKE 'NAS%' THEN SUBSTRING(CID, 4, len(CID))
     ELSE CID
END CID,
CASE WHEN CAST (BDATE AS DATE) > CURRENT_DATE() THEN NULL  
     ELSE CAST (BDATE AS DATE)
END BDATE,
CASE WHEN UPPER(TRIM(GEN)) IN ('F', 'FEMALE') THEN 'Female'
     WHEN UPPER(TRIM(GEN)) IN ('M', 'MALE') THEN 'Male'
     ELSE 'n/a'
END GEN
FROM bronze.erp_cust_az12

In [0]:
%sql
SELECT 
REPLACE(cid,'-',''),  -- remove dash to match with cust_info table 
CASE WHEN TRIM(cntry) IN ('USA', 'US') THEN 'United State'   
     WHEN TRIM(cntry) = '' OR cntry is NULL THEN 'n/a'
     WHEN TRIM(cntry) = 'DE' THEN 'Germany'
     ELSE cntry 
END cntry

FROM bronze.erp_loc_a101
WHERE REPLACE(cid, '-', '') NOT IN 
(SELECT cst_key FROM silver.crm_cust_info);   -- check if there's any unmatching value between custInfo and erp_loc_a101 
-- Result: no unmatched value 


In [0]:
%sql
SELECT UPPER(TRIM(id)) AS id,
       TRIM(cat) AS CAT, 
       TRIM(subcat) AS SUBCAT, 
       TRIM(maintenance) AS MAINTENANCE
FROM bronze.erp_px_cat_g1v2;

 
 
SELECT id FROM bronze.erp_px_cat_g1v2 
WHERE id NOT IN 
(SELECT cat_id FROM silver.crm_prd_info); 

SELECT * FROM bronze.erp_px_cat_g1v2 WHERE TRIM(cat) != cat OR TRIM(subcat) != subcat OR TRIM(maintenance) != maintenance; -- check unwanted space 
    
-- data standardization
SELECT distinct cat, subcat, maintenance from bronze.erp_px_cat_g1v2